## Camada 5 (bônus) - evals: o TDD do mundo não-determinístico

Um LLM no meio não elimina o teste: muda o formato. Casos com
gabarito, rodados com score. Mudou o prompt? Trocou o modelo? Roda
de novo a suíte. O nome disso: *eval-driven development*.

A suíte completa roda 22 chamadas (11 casos x 2 prompts); só o
zero-shot (padrão abaixo) já são 11 e já valem a demo.


In [ ]:
# Setup encapsulado: cada notebook é auto-contido (a ordem ENTRE
# notebooks é livre; dentro dele, as células rodam de cima para baixo)
import os, pathlib, sys
sys.path.append(str(pathlib.Path.cwd().parent))

import pandas as pd
from llm import chat, usage_total

print(f"Modelo: {os.environ.get('LLM_MODEL', '?')}")
df = pd.read_csv('../data/reclamacoes.csv')
CATEGORIAS = 'entrega | cobranca | produto | atendimento | app'

def prompt_zero_shot(texto: str) -> str:
    return (
        'Classifique a reclamacao em UMA das categorias:\n'
        f'{CATEGORIAS}\n\n'
        f'Reclamacao: "{texto}"\n\n'
        'Responda apenas com a categoria, em minusculas, e nada mais.'
    )


In [ ]:
# Os quatro exemplos abaixo são âncoras de regra escolhidas à mão:
# cada um ensina 'olhe além da palavra-chave: a causa raiz decide'.
# Escolher bons exemplos é o novo feature engineering: qualidade > quantidade.
FEW_SHOT = [
    ('O atendente disse que o reembolso do pedido que nao chegou foi aprovado e o boleto segue vindo', 'entrega'),
    ('A segunda parcela do pedido ficou mais cara que o combinado no carrinho', 'cobranca'),
    ('A camera parou de funcionar uma semana apos a compra', 'produto'),
    ('O atendente desligou o telefone na cara da cliente depois de 40 minutos de fila', 'atendimento'),
]

def prompt_few_shot(texto: str) -> str:
    exemplos = '\n'.join(f'Reclamacao: "{t}" -> {c}' for t, c in FEW_SHOT)
    return (
        f'{exemplos}\n\n'
        'Classifique pela CAUSA RAIZ da reclamacao, em UMA das categorias:\n'
        f'{CATEGORIAS}\n\n'
        f'Reclamacao: "{texto}"\n\n'
        'Responda apenas com a categoria, em minusculas, e nada mais.'
    )


In [ ]:
# A suíte: gabarito por caso, score agregado no fim da rodada
CASOS = [(df.iloc[i]['texto'], df.iloc[i]['categoria']) for i in [0, 1, 2, 3, 4, 5, 6, 7, 15, 16, 17]]

def suite(prompt_builder, casos=CASOS):
    acertos = 0
    for texto, esperado in casos:
        pred = chat([{'role': 'user', 'content': prompt_builder(texto)}], temperature=0).strip().lower()
        acerto = pred == esperado
        acertos += acerto
        print(f"[{'OK' if acerto else 'ERROU'}] esperado={esperado:12} veio={pred[:20]}")
    return acertos / len(casos)

# As duas taxas lado a lado contam a história: mesmo modelo, mesma
# tarefa - o que mudou foi o prompt, e o score acompanha
print()
print(f'Zero-shot: {suite(prompt_zero_shot):.0%}')
# print(f'Few-shot : {suite(prompt_few_shot):.0%}')
print('Tokens acumulados:', usage_total())
